cifar100数据之前已经处理过了(已经将无标签样本划分出来了) path: DL/data/cifar100
1. 读取数据

In [1]:
import logging
import os
import pickle
import tarfile

import torchvision.transforms as transforms
import numpy as np
import torch


workspacedir = os.getcwd()
c100_dir = os.path.join(workspacedir, "data\\cifar100")                                 # cifar100数据根目录
c100_tar_gz = os.path.join(workspacedir, "data\\cifar100\\cifar-100-python.tar.gz")     # cifar100源文件路径
c100_un_tar = os.path.join(workspacedir, "data\\cifar100\\cifar-100-python")             # cifar100解压后路径

# --------------------
# 加载cifar100数据
# --------------------

def un_tar(file_path, extract_to=".") -> None:
    """解压tar.gz到目标目录"""
    if not os.path.exists(extract_to):
        os.makedirs(extract_to)
    with tarfile.open(file_path, 'r:gz') as tar:
        for member in tar.getmembers():
            tar.extract(member, extract_to)
            logging.info(f"extracted {member.name} to {extract_to}")

def parse_pickle(file) -> dict:
    """
    解析pickle文件
    """
    with open(file, 'rb') as fo:
        dct = pickle.load(fo, encoding='bytes')
    return dct


def load_c100_batch(file_path):
    """
    处理单个pickle文件, 返回图像和标签
    """
    batch = parse_pickle(file_path)

    images = batch[b'data']           # shape (50000, 3072)
    labels = batch[b'fine_labels']    # len 50000

    images = images.reshape(-1, 3, 32, 32)      # 重塑图像为[N, 3, 32, 32]
    images = images.astype(np.float32) / 255.0  # 归一化[0, 1]

    images = torch.tensor(images)
    labels = torch.tensor(labels, dtype=torch.long)

    return images, labels

def load_c100_data(data_dir):
    """
    加载cifar100数据
    meta: dict_keys([b'fine_labels'-细粒度标签索引])
    train/test: dict_keys([b'filenames', b'batch_label', b'fine_labels'-细粒度标签索引, b'coarse_labels'-粗粒度标签索引, b'data'])

    Args:
        data_dir: cifar100解压后的路径
    """

    c100_train = os.path.join(data_dir, "train")
    c100_test = os.path.join(data_dir, "test")

    images_train, labels_train = load_c100_batch(c100_train)            # train
    images_test, labesl_test = load_c100_batch(c100_test)               # test

    return images_train, labels_train, images_test, labesl_test

# 1.解压数据
# un_tar(c100_tar_gz, c100_dir)
# 2.加载数据
images_train_all, labels_train_all, images_test_all, labels_test_all = load_c100_data(c100_un_tar)
print(f"训练集图像数量: {images_train_all.shape[0]}")
print(f"测试集图像数量: {images_test_all.shape[0]}")

训练集图像数量: 50000
测试集图像数量: 10000


In [2]:
from collections import defaultdict
import random
from torch.utils.data import DataLoader
from torchvision import transforms

# -------------------------
# 拆分"有标签”和"无标签"数据
# -------------------------
NUM_LABELED_PER_CLASS = 50  # 每类有标签样本数
NUM_CLASSES = 10

# 初始化计数器
label_counts = defaultdict(int)

labeled_images = []
labeled_labels = []
unlabeled_images = []
unlabeled_labels = []

# 打乱索引以确保随机性
indices = list(range(len(images_train_all)))
random.shuffle(indices)

for idx in indices:
    img = images_train_all[idx]
    lbl = labels_train_all[idx].item()

    if label_counts[lbl] < NUM_LABELED_PER_CLASS:
        labeled_images.append(img)
        labeled_labels.append(lbl)
        label_counts[lbl] += 1
    else:
        unlabeled_images.append(img)
        unlabeled_labels.append(lbl)  # 标签仍然存在，但后续不使用

print(f"有标签数据数量: {len(labeled_images)}")
print(f"无标签数据数量: {len(unlabeled_images)}")

有标签数据数量: 5000
无标签数据数量: 45000


In [3]:
from torch.utils.data import Dataset

class LabeledCIFARDataset(Dataset):
    """
    带标签数据集
    """
    def __init__(self, images, labels, transform=None):
        """
        有标签数据集
        """
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        x = self.images[idx]
        y = self.labels[idx]
        if self.transform:
            x = self.transform(x)
        return x, y


class UnlabeledCIFARDataset(Dataset):
    def __init__(self, images, transform=None):
        """
        无标签数据集
        """
        self.images = images
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        x = self.images[idx]
        if self.transform:
            x = self.transform(x)
        return x


class CIFARValDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        """
        验证/测试数据集
        """
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        x = self.images[idx]
        y = self.labels[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

# ---------------------
# 创建dataloader
# ---------------------

BATCH_SIZE = 32

mean = [0.5071, 0.4867, 0.4408]  # CIFAR-100数据集RGB三个通道的平均值
std = [0.2675, 0.2565, 0.2761]   # CIFAR-100数据集RGB三个通道的标准差

transform = transforms.Compose([
    transforms.Normalize(mean, std)
])

# 有标签 DataLoader
labeled_dataset = LabeledCIFARDataset(labeled_images, labeled_labels, transform)
labeled_loader = DataLoader(
    labeled_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

# 无标签 DataLoader
unlabeled_dataset = UnlabeledCIFARDataset(unlabeled_images, transform)
unlabeled_loader = DataLoader(
    unlabeled_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

# 验证/测试 DataLoader
val_dataset = CIFARValDataset(images_test_all, labels_test_all)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(f"有标签数据批次数: {len(labeled_loader)}")
print(f"无标签数据批次数: {len(unlabeled_loader)}")
print(f"验证/测试数据批次数: {len(val_loader)}")

有标签数据批次数: 156
无标签数据批次数: 1406
验证/测试数据批次数: 313


In [4]:
# ------------------------
# 训练(伪标签)
# ------------------------
from models.resnet_kaming import ResNet34, ResNet50
from torch import nn, optim
from itertools import cycle

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')   # 设备类型
print(f"device: {device}")

resnet34 = ResNet34(num_class=100).to(device=device)   # 初始化模型
resnet50 = ResNet50(num_class=100).to(device=device)   # 初始化模型


def evaluate_on_val(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x_val, y_val in test_loader:
            x_val = x_val.to(device)
            y_val = y_val.to(device)

            output = model.forward(x_val)

            pres = torch.argmax(output, dim=1)
            correct += (pres == y_val).sum().item()
            total += y_val.size(0)
    acc = correct / total
    return acc

def train_supervised_only(model, labeled_loader, val_loader, epochs=10, lr=1e-1):
    # optimizer = optim.Adam(model.parameters(), lr=lr)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        for x, y in labeled_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
        
        scheduler.step()
        
        val_acc = evaluate_on_val(model, val_loader)
        print(f"Pure Supervised Epoch {epoch+1}, lr: {lr:.6f} loss: {loss:.4f} Val Acc: {val_acc:.4f}")


def train_semi_supervised(model, labeled_loader, unlabeled_loader, val_loader, epochs=20, lr=1e-1, lambda_unsupervised=1.0, confidence_threshold=0.6):
    """
    半监督训练
    Args:
        model: 模型
        labeled_loader: 带标签样本
        unlabeled_loader: 无标签样本
        val_loader: 测试样本
        epochs: 训练批次
        lr: 学习率
    """
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    # optimizer = optim.Adam(model.parameters(), lr=lr)
    cross_entropy_loss = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()

        # n_l = len(labeled_loader)
        # n_u = len(unlabeled_loader)
        # norm = math.pow(n_u + n_l, 2)

        total_loss = 0.0                # 总损失
        total_cross_loss = 0.0          # 交叉熵损失
        # total_con_loss = 0.0            # 一致性损失
        # total_smooth_loss = 0.0         # 平滑性损失
        total_unsupervised = 0.0        # 无监督总损失(= 一致性损失 + 平滑性损失)
        processed_batches = 0

        labeled_iter = cycle(labeled_loader)
        unlabeled_iter = iter(unlabeled_loader)

        # 无标签数据训练为基准
        for x_unlabeled in unlabeled_loader:
            # 获取带标签数据
            try:
                x_labeled, y_labeled = next(labeled_iter)
            except StopIteration:
                labeled_iter = cycle(labeled_loader)
                x_labeled, y_labeled = next(labeled_iter)
            
            # 转移数据到设备
            x_labeled = x_labeled.to(device)
            y_labeled = y_labeled.to(device)
            x_unlabeled = x_unlabeled.to(device)

            optimizer.zero_grad()

            # 带标签数据的前向传播
            pred_labeled = model(x_labeled)
            cross_loss = cross_entropy_loss(pred_labeled, y_labeled)
            # 无标签数据前向传播
            with torch.no_grad():  # 伪标签不反向传播
                logits_unlabeled = model(x_unlabeled)
                probs = torch.softmax(logits_unlabeled, dim=1)
                max_probs, pseudo_labels = torch.max(probs, dim=1)
                # 只使用高置信度的伪标签
                high_confidence_mask = max_probs >= confidence_threshold

            # 计算无监督损失
            loss_unsupervised  = 0.0
            if high_confidence_mask.sum().item() > 0:
                pseudo_labels = pseudo_labels[high_confidence_mask]
                logits_unlabeled_hcm = logits_unlabeled[high_confidence_mask]
                loss_unsupervised  = cross_entropy_loss(logits_unlabeled_hcm, pseudo_labels)
            
            # 合并损失反向传播
            total_batch_loss = cross_loss + lambda_unsupervised * loss_unsupervised 
            total_batch_loss.backward()
            optimizer.step()

            # 统计损失
            total_loss += total_batch_loss.item()
            total_cross_loss += cross_loss.item()  
            total_unsupervised += loss_unsupervised.item() if high_confidence_mask.sum() > 0 else 0.0
            processed_batches += 1

        # 学习率调整
        scheduler.step()

        # 计算平均损失
        avg_loss = total_loss / processed_batches
        avg_cross = total_cross_loss / processed_batches
        avg_unsupervised = total_unsupervised / processed_batches

        # 测试集评估
        val_acc = evaluate_on_val(model, val_loader)
        print(f"Epoch {epoch+1}/{epochs}, "
              f"Loss: {avg_loss:.4f}, "
              f"Cross: {avg_cross:.4f}, "
              f"Unsupervised: {avg_unsupervised:.4f}, "
              f"Val Acc: {val_acc:.4f}")


train_semi_supervised(resnet34, labeled_loader, unlabeled_loader, val_loader)
# 测试纯监督训练
# train_supervised_only(resnet34, labeled_loader, val_loader, epochs=10)

device: cuda
Epoch 1/20, Loss: 4.4627, Cross: 4.4260, Unsupervised: 0.0367, Val Acc: 0.0230


KeyboardInterrupt: 

In [ ]:
# ----------------------
#   训练（逐点流形正则化）
# ----------------------

from models.resnet_kaming import ResNet34, ResNet50
from torch import nn, optim
from itertools import cycle

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')   # 设备类型
print(f"device: {device}")

resnet34 = ResNet34(num_class=100).to(device=device)   # 初始化模型
resnet50 = ResNet50(num_class=100).to(device=device)   # 初始化模型


def evaluate_on_val(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x_val, y_val in test_loader:
            x_val = x_val.to(device)
            y_val = y_val.to(device)

            output = model.forward(x_val)

            pres = torch.argmax(output, dim=1)
            correct += (pres == y_val).sum().item()
            total += y_val.size(0)
    acc = correct / total
    return acc

def train_supervised_only(model, labeled_loader, val_loader, epochs=10, lr=1e-1):
    # optimizer = optim.Adam(model.parameters(), lr=lr)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        for x, y in labeled_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
        
        scheduler.step()
        
        val_acc = evaluate_on_val(model, val_loader)
        print(f"Pure Supervised Epoch {epoch+1}, lr: {lr:.6f} loss: {loss:.4f} Val Acc: {val_acc:.4f}")


def train_semi_supervised(model, labeled_loader, unlabeled_loader, val_loader, epochs=20, lr=1e-1, lambda_unsupervised=1.0, confidence_threshold=0.6):
    """
    半监督训练
    Args:
        model: 模型
        labeled_loader: 带标签样本
        unlabeled_loader: 无标签样本
        val_loader: 测试样本
        epochs: 训练批次
        lr: 学习率
    """
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    # optimizer = optim.Adam(model.parameters(), lr=lr)
    cross_entropy_loss = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()

        # n_l = len(labeled_loader)
        # n_u = len(unlabeled_loader)
        # norm = math.pow(n_u + n_l, 2)

        total_loss = 0.0                # 总损失
        total_cross_loss = 0.0          # 交叉熵损失
        # total_con_loss = 0.0            # 一致性损失
        # total_smooth_loss = 0.0         # 平滑性损失
        total_unsupervised = 0.0        # 无监督总损失(= 一致性损失 + 平滑性损失)
        processed_batches = 0

        labeled_iter = cycle(labeled_loader)
        unlabeled_iter = iter(unlabeled_loader)

        # 无标签数据训练为基准
        for x_unlabeled in unlabeled_loader:
            # 获取带标签数据
            try:
                x_labeled, y_labeled = next(labeled_iter)
            except StopIteration:
                labeled_iter = cycle(labeled_loader)
                x_labeled, y_labeled = next(labeled_iter)
            
            # 转移数据到设备
            x_labeled = x_labeled.to(device)
            y_labeled = y_labeled.to(device)
            x_unlabeled = x_unlabeled.to(device)

            optimizer.zero_grad()

            # 带标签数据的前向传播
            pred_labeled = model(x_labeled)
            cross_loss = cross_entropy_loss(pred_labeled, y_labeled)
            # 无标签数据前向传播
            with torch.no_grad():  # 伪标签不反向传播
                logits_unlabeled = model(x_unlabeled)
                probs = torch.softmax(logits_unlabeled, dim=1)
                max_probs, pseudo_labels = torch.max(probs, dim=1)
                # 只使用高置信度的伪标签
                high_confidence_mask = max_probs >= confidence_threshold

            # 计算无监督损失
            loss_unsupervised  = 0.0
            if high_confidence_mask.sum().item() > 0:
                pseudo_labels = pseudo_labels[high_confidence_mask]
                logits_unlabeled_hcm = logits_unlabeled[high_confidence_mask]
                loss_unsupervised  = cross_entropy_loss(logits_unlabeled_hcm, pseudo_labels)
            
            # 合并损失反向传播
            total_batch_loss = cross_loss + lambda_unsupervised * loss_unsupervised 
            total_batch_loss.backward()
            optimizer.step()

            # 统计损失
            total_loss += total_batch_loss.item()
            total_cross_loss += cross_loss.item()  
            total_unsupervised += loss_unsupervised.item() if high_confidence_mask.sum() > 0 else 0.0
            processed_batches += 1

        # 学习率调整
        scheduler.step()

        # 计算平均损失
        avg_loss = total_loss / processed_batches
        avg_cross = total_cross_loss / processed_batches
        avg_unsupervised = total_unsupervised / processed_batches

        # 测试集评估
        val_acc = evaluate_on_val(model, val_loader)
        print(f"Epoch {epoch+1}/{epochs}, "
              f"Loss: {avg_loss:.4f}, "
              f"Cross: {avg_cross:.4f}, "
              f"Unsupervised: {avg_unsupervised:.4f}, "
              f"Val Acc: {val_acc:.4f}")


train_semi_supervised(resnet34, labeled_loader, unlabeled_loader, val_loader)
# 测试纯监督训练
# train_supervised_only(resnet34, labeled_loader, val_loader, epochs=10)

In [5]:
import torch
print(torch.__version__)

2.5.1
